In [1]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.data.iloc[index, 1]  # 'Image_paths' is the column containing file paths
        img_path = img_path.replace("E:\\TARUN\\Projects\\Autism Detection\\Data\\data_png",'/kaggle/input/autism')
        img_path = img_path.replace("\\",'/')

        image = Image.open(img_path)
        
        # Convert single-channel image to RGB
        if image.mode == 'L':
            image = image.convert('RGB')

        label = int(self.data.iloc[index, 3])  # 'LABEL' is the column containing labels (0 or 1)

        if self.transform:
            image = self.transform(image)
        
        # Convert PIL image to a NumPy array
        image = np.array(image)

        return image, label

# Load your dataset without normalization for the mean and std calculation
raw_dataset = CustomDataset('/kaggle/input/autism-csv/extracted_random_labels.csv', transform=None)
# Initialize lists to store channel-wise mean and standard deviation
channel_means = [0.0, 0.0, 0.0]
channel_stds = [0.0, 0.0, 0.0]

# Iterate through your dataset to calculate channel-wise mean and std
for sample in raw_dataset:
    image, _ = sample  # Assuming the label is not used for this calculation
    for i in range(3):  # Assuming your images have 3 channels (R, G, B)
        channel_means[i] += image[i, :, :].mean()
        channel_stds[i] += image[i, :, :].std()

# Calculate the mean and std values across the entire dataset
num_samples = len(raw_dataset)
mean = [c_mean / num_samples for c_mean in channel_means]
std = [c_std / num_samples for c_std in channel_stds]

# Define the transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224 pixels
    transforms.ToTensor(),  # Convert to a PyTorch tensor
    transforms.Normalize(mean, std)  # Normalize using the calculated mean and std
])

# Define batch size
batch_size = 64

# Create train and test data loaders
train_dataset = CustomDataset('/kaggle/input/autism-csv/extracted_random_labels_train.csv', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = CustomDataset('/kaggle/input/autism-csv/extracted_random_labels_test.csv', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

val_dataset = CustomDataset('/kaggle/input/autism-csv/extracted_random_labels_validation.csv', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size)


In [2]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Test dataset size: {len(val_dataset)}")

Train dataset size: 72367
Test dataset size: 20102
Test dataset size: 8041


In [3]:
import torch
import timm
import torch.nn as nn
import torch.optim as optim

# Define the pre-trained ViT model (e.g., ViT-B/16)
model = timm.create_model('vit_base_patch16_224', pretrained=True)

# Modify the model's head for binary classification
num_classes = 1  # Change this to match the number of classes in your dataset
model.head = nn.Linear(in_features=model.head.in_features, out_features=num_classes)

# Define the loss function
criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss

# Define the hyperparameters for the Adam optimizer
learning_rate = 0.001  # η
beta1 = 0.9  # β1
beta2 = 0.999  # β2
epsilon = 1e-7  # ε

# Create an instance of the Adam optimizer with the specified hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(beta1, beta2), eps=epsilon)

# Move the model to the GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [3]:
import torch
import timm
import torch.nn as nn
import torch.optim as optim

# Define the smaller pre-trained ViT model (e.g., ViT-Small-Patch16)
model = timm.create_model('vit_small_patch16_224', pretrained=True)  # Use 'vit_small_patch16_224'

# Modify the model's head for binary classification
num_classes = 1  # Change this to match the number of classes in your dataset
model.head = nn.Linear(in_features=model.head.in_features, out_features=num_classes)

# Define the loss function
criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss

# Define the hyperparameters for the Adam optimizer
learning_rate = 0.001  # η
beta1 = 0.9  # β1
beta2 = 0.999  # β2
epsilon = 1e-7  # ε

# Create an instance of the Adam optimizer with the specified hyperparameters
optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(beta1, beta2), eps=epsilon)

# Move the model to the GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [5]:
import torch
import timm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# Load your pretrained model weights here if needed
# model.load_state_dict(torch.load('Adam_full_weights.pth'))

# Create empty lists to store training and validation losses and accuracies
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Set the current epoch number to continue from where you left off
current_epoch = 0  # Change this to the epoch you want to continue from

num_epochs = 3  # Set the total number of epochs you want to run

# Define early stopping parameters
patience = 5  # Number of epochs with no improvement after which training will stop
min_delta = 0.001  # Minimum change in validation loss to be considered as improvement
best_val_loss = float('inf')
epochs_without_improvement = 0

# Continue training for additional epochs
for epoch in range(current_epoch, num_epochs):
    model.train()
    running_train_loss = 0.0
    train_preds = []
    train_targets = []

    # Training loop with a progress bar
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs} (Training)'):
        images, labels = images.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.float())  # Use binary cross-entropy loss

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        predicted = (outputs > 0).float()  # Convert to 0 or 1
        train_preds.extend(predicted.cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    # Calculate and print the average training loss for this epoch
    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Calculate training accuracy for this epoch
    train_accuracy = accuracy_score(train_targets, train_preds)
    train_accuracies.append(train_accuracy)

    model.eval()  # Set the model to evaluation mode
    running_val_loss = 0.0
    val_preds = []
    val_targets = []

    # Validation loop with a progress bar
    for images, labels in tqdm(val_loader, desc=f'Epoch {epoch + 1}/{num_epochs} (Validation)'):
        images, labels = images.to(device), labels.to(device)  # Move data to GPU
        outputs = model(images)
        loss = criterion(outputs, labels.float())  # Use binary cross-entropy loss

        running_val_loss += loss.item()

        predicted = (outputs > 0).float()  # Convert to 0 or 1
        val_preds.extend(predicted.cpu().numpy())
        val_targets.extend(labels.cpu().numpy())

    # Calculate and print the average validation loss for this epoch
    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Calculate validation accuracy for this epoch
    val_accuracy = accuracy_score(val_targets, val_preds)
    val_accuracies.append(val_accuracy)

    # Early stopping code remains the same
    if avg_val_loss + min_delta < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), '/kaggle/working/Adam_vit_weights.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping after {patience} epochs without improvement.')
            break

    # Print loss and accuracy for this epoch
    print(f'Epoch [{epoch + 1}/{num_epochs}], Training Loss: {avg_train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}')
    print(f'Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}')

print('Finished Training')


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
Epoch 1/3 (Training):   0%|          | 0/1131 [00:05<?, ?it/s]


ValueError: Target size (torch.Size([64])) must be the same as input size (torch.Size([64, 1]))

In [6]:
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.float())  # Convert labels to float for BCEWithLogitsLoss
        loss.backward()
        optimizer.step()

    # Validation loop (optional)
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # Calculate and store validation metrics as needed



OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB (GPU 0; 14.76 GiB total capacity; 13.49 GiB already allocated; 115.75 MiB free; 13.68 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF